In [5]:
from rdkit import Chem
from rdkit.Chem import rdRascalMCES
from lib import MCES_ILP
from graph_construction import construct_graph
from bounds import filter2
import polars as pl
import os
# generat
data_file_path = "/home/analytit_admin/dev/MS_utils/ms_utils/mces/dsstox_smiles_medium.csv"
number_of_mol:int = 10
smiles = pl.scan_csv(data_file_path).head(number_of_mol).collect().to_series().to_list()

In [35]:
# now calculate mces usign rascal mces for all 10 molecules
mols = [Chem.MolFromSmiles(smi) for smi in smiles]
# make sure all molecules are valid
mols = [mol for mol in mols if mol is not None]
# use loop on pairs of molecules
mces = []
opts = rdRascalMCES.RascalOptions()
opts.similarityThreshold = 0.0
opts.singleLargestFrag = True
for i in range(len(mols)):
    for j in range(i + 1, len(mols)):
        res= rdRascalMCES.FindMCES(mols[i], mols[j], opts)
        if res:
            mces_res = res[0]
            
            # Get bond matches from the result
            bond_matches = mces_res.bondMatches()
            mol1_matched_bonds_indices = {match[0] for match in bond_matches}
            mol2_matched_bonds_indices = {match[1] for match in bond_matches}

            # Calculate cost for matched bonds (difference in bond orders)
            cost_matched = 0
            for mol1_idx, mol2_idx in bond_matches:
                bond1 = mols[i].GetBondWithIdx(mol1_idx)
                bond2 = mols[j].GetBondWithIdx(mol2_idx)
                cost_matched += abs(bond1.GetBondTypeAsDouble() - bond2.GetBondTypeAsDouble())

            # Calculate cost for unmatched bonds in mol1
            cost_unmatched1 = 0
            for b_idx in range(mols[i].GetNumBonds()):
                if b_idx not in mol1_matched_bonds_indices:
                    cost_unmatched1 += mols[i].GetBondWithIdx(b_idx).GetBondTypeAsDouble()
            
            # Calculate cost for unmatched bonds in mol2
            cost_unmatched2 = 0
            for b_idx in range(mols[j].GetNumBonds()):
                if b_idx not in mol2_matched_bonds_indices:
                    cost_unmatched2 += mols[j].GetBondWithIdx(b_idx).GetBondTypeAsDouble()

            # The total graph distance is the sum of these costs
            graph_distance = cost_matched + cost_unmatched1 + cost_unmatched2

            # save this into a list, don't print it
            mces.append({
                "mol1": i,
                "mol2": j,
                "graph_distance": graph_distance,
                "mces_similarity": mces_res.similarity
            })
            


In [ ]:
# now we calculate the mces using the ILP method- converting all to a graph
graphs = [construct_graph(smile) for smile in smiles]
# loop on pairs of graphs
mces_ilp = []
for i in range(len(graphs)):
    for j in range(i + 1, len(graphs)):
        res = MCES_ILP(graphs[i], graphs[j],no_ilp_threshold=True,threshold=100,solver="gurobi")
        if res:
            mces_ilp.append({
                "mol1": i,
                "mol2": j,
                "graph_distance": res[0],
            })


Gurobi Optimizer version 12.0.3 build v12.0.3rc0 (linux64 - "Ubuntu 24.04.1 LTS")

CPU model: 13th Gen Intel(R) Core(TM) i9-13900K, instruction set [SSE2|AVX|AVX2]
Thread count: 32 physical cores, 32 logical processors, using up to 32 threads

Academic license 2647979 - for non-commercial use only - registered to ni___@weizmann.ac.il
Optimize a model with 2308 rows, 1549 columns and 9222 nonzeros
Model fingerprint: 0x14cb7b40
Variable types: 0 continuous, 1549 integer (0 binary)
Coefficient statistics:
  Matrix range     [1e+00, 1e+00]
  Objective range  [5e-01, 2e+00]
  Bounds range     [1e+00, 1e+00]
  RHS range        [1e+00, 1e+00]
Found heuristic solution: objective 83.0000000
Presolve removed 829 rows and 138 columns
Presolve time: 0.01s
Presolved: 1479 rows, 1411 columns, 8222 nonzeros
Variable types: 0 continuous, 1411 integer (1411 binary)

Root relaxation: objective 2.729316e+01, 9031 iterations, 0.40 seconds (0.76 work units)

    Nodes    |    Current Node    |     Objectiv

In [38]:

# compare the two results
comparison = []
for mces_res in mces:
    for mces_ilp_res in mces_ilp:
        if mces_res["mol1"] == mces_ilp_res["mol1"] and mces_res["mol2"] == mces_ilp_res["mol2"]:
            comparison.append({
                "mol1": mces_res["mol1"],
                "mol2": mces_res["mol2"],
                "graph_distance_mces": mces_res["graph_distance"],
                "mces_similarity_mces": mces_res["mces_similarity"],
                "graph_distance_mces_ilp": mces_ilp_res["graph_distance"],
            })

In [39]:
print("Comparison of MCES results:")
for comp in comparison:
    print(f"Mol1: {comp['mol1']}, Mol2: {comp['mol2']}, "
          f"Graph Distance (MCES): {comp['graph_distance_mces']}, "
          f"MCES Similarity: {comp['mces_similarity_mces']}, "
          f"Graph Distance (MCES ILP): {comp['graph_distance_mces_ilp']}")

Comparison of MCES results:
Mol1: 0, Mol2: 1, Graph Distance (MCES): 59.0, MCES Similarity: 0.07439724454649828, Graph Distance (MCES ILP): 29.0
Mol1: 0, Mol2: 2, Graph Distance (MCES): 56.0, MCES Similarity: 0.061627347135291284, Graph Distance (MCES ILP): 28.0
Mol1: 0, Mol2: 3, Graph Distance (MCES): 55.0, MCES Similarity: 0.05223880597014925, Graph Distance (MCES ILP): 29.0
Mol1: 0, Mol2: 4, Graph Distance (MCES): 62.5, MCES Similarity: 0.04795693662833374, Graph Distance (MCES ILP): 30.5
Mol1: 0, Mol2: 5, Graph Distance (MCES): 47.0, MCES Similarity: 0.11156083986845435, Graph Distance (MCES ILP): 33.0
Mol1: 0, Mol2: 6, Graph Distance (MCES): 43.0, MCES Similarity: 0.0979716800612323, Graph Distance (MCES ILP): 25.0
Mol1: 0, Mol2: 7, Graph Distance (MCES): 59.5, MCES Similarity: 0.05878300803673938, Graph Distance (MCES ILP): 27.5
Mol1: 0, Mol2: 8, Graph Distance (MCES): 49.5, MCES Similarity: 0.06648575305291723, Graph Distance (MCES ILP): 29.5
Mol1: 0, Mol2: 9, Graph Distance (MC

conclusions: 
the reason for the discrepency, based on copilots explanation, is that the mapping of the MCES_ILP is optimal to minimize the edit ditance, but that of rascal mces is not. hence, it can't be used to calculate the edit distance.